# Using Label Studio to create a dataset for training

## Install Label Studio

First install Label Studio, which is the tool used for labeling datasets for object classification models, such as YOLO. 

To install Label Studio input the following command into your CLI: `pip install label-studio`

To open Label Studio input the following command into your CLI: `label-studio`

After labeling your data be sure to export it in the YOLOv8 OBB format

# Split Data

This Jupyter Notebook is used to split a large sum of labeled data into smaller subsets used for training, testing, and validating a self trained YOLO OBB model. To begin copy the contents for your exported zip file from Label Studio and paste them into the 'unsplit_data' directory of this project. Your directory should look like this:

unsplit_data/  
│── images/  
│   ├── image1.jpg/.jpeg/.png  
│   ├── image2.jpg/.jpeg/.png  
│   ├── image3.jpg/.jpeg/.png  
│   ├── ...  
│  
│── labels/  
│   ├── image1.txt  
│   ├── image2.txt  
│   ├── image3.txt  
│   ├── ...  
│  
│── notes.json  
│── classes.txt  


In [34]:
#Necessary imports for completing the 'Split Data' portion of the script
import os
import shutil
import random

In [37]:
# Define paths to your data
source_images = os.getcwd() + "/datasets/unsplit_data/images"
source_lables = os.getcwd() + "/datasets/unsplit_data/labels"
output_directory = "datasets"

In [38]:
# Create train, val, test folders
sets = ["train", "val", "test"]
subdirs = ["images", "labels"]

In [39]:
for s in sets:
    for sub in subdirs:
        os.makedirs(os.path.join(output_directory, s, sub), exist_ok=True)

In [40]:
# Get all image files
image_files = [f for f in os.listdir(source_images) if f.endswith(('.jpg', '.png', '.jpeg'))]

In [41]:
# Shuffle and split
random.shuffle(image_files)
num_images = len(image_files)

train_split = int(num_images * 0.8)
val_split = int(num_images * 0.1)

train_files = image_files[:train_split]
val_files = image_files[train_split:train_split + val_split]
test_files = image_files[train_split + val_split:]

In [42]:
# Function to move files
def move_files(file_list, split):
    for file in file_list:
        img_src = os.path.join(source_images, file)
        label_src = os.path.join(source_lables, file.replace(os.path.splitext(file)[1], ".txt"))

        img_dst = os.path.join(output_directory, split, "images", file)
        label_dst = os.path.join(output_directory, split, "labels", file.replace(os.path.splitext(file)[1], ".txt"))

        if os.path.exists(img_src):
            shutil.copy(img_src, img_dst)
        if os.path.exists(label_src):
            shutil.copy(label_src, label_dst)

In [43]:
# Move files to respective folders
move_files(train_files, "train")
move_files(val_files, "val")
move_files(test_files, "test")

print("Dataset successfully organized!")

Dataset successfully organized!


# Creation of the YAML File for Training

When exporting your images and annotationd from file studio your project zip folder should have also had two other files in them, named 

In [51]:
num_classes = 0
classes = []
with open(os.getcwd() + '/datasets/unsplit_data/classes.txt', 'r') as file:
    lines = file.readlines()
    for line in lines:
        num_classes += 1
        classes.append(line.strip())
        print(line.strip())  # strip() removes leading/trailing whitespace
names_dict = {i: name for i, name in enumerate(classes)}

return-bend


In [46]:
import yaml

In [57]:
data = {
    'train': os.getcwd() + 'datasets/train/images',
    'val': os.getcwd() + 'datasets/val/images',
    'test': os.getcwd() + 'datasets/test/images',
    'train_labels': os.getcwd() + 'datasets/train/labels',
    'test_labels' : os.getcwd() + 'datasets/test/labels/',
    'val_labels' : os.getcwd() + 'datasets/val/labels/',
    'nc' : num_classes,
    'names': names_dict
}

In [58]:
with open('sample.yaml', 'w') as file:
    yaml.dump(data, file, default_flow_style=False)
print("YAML file created successfully")

YAML file created successfully


# Train the YOLO OBB Model

<span style="color:red; font-size:40px;">WARNING - THIS CAN TAKE A VERY LONG TIME</span>

If your dataset is large I recommend using an AWS EC2 instance with a GPU to train this model.



In [ ]:
from ultralytics import YOLO

# Define paths
DATA_YAML = os.getcwd() + "/datasets/unsplit_data/sample.yaml"  # Path to your dataset YAML file
MODEL_CHECKPOINT = "yolov8n-obb.pt"  # You can change this to 'yolov8m-obb.pt' or other versions

# Load the model (pretrained)
model = YOLO(MODEL_CHECKPOINT)

# Train the model
model.train(
    data=DATA_YAML,
    epochs=100,  # Adjust the number of training epochs
    imgsz=1024,  # Image size for training
    batch=8,  # Adjust batch size based on your GPU memory
    device="mps",  # Use "cpu" if training on a CPU, "cuda" for GPU (if available)
    workers=4,  # Number of data loading workers
    project="obb_training",  # Output directory for runs
    name="train_run",  # Experiment name
    save_period=10,  # Save checkpoints every 10 epochs
    verbose=True  # Show training logs
)

# Evaluate the model on validation set
metrics = model.val()

# Export model to ONNX and TensorFlow formats (optional)
#model.export(format="onnx")
#model.export(format="tflite")
